<a href="https://colab.research.google.com/github/weagan/Speculative-Decoding/blob/main/target_drafter_combo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dual‑GPU Speculative Decoding

Target model: **meta-llama/Llama-3.1-8B** (GPU 0)
Draft model: **meta-llama/Llama-3.2-1B** (GPU 1)

Communication between models uses **text only**, not token tensors.

Before speculative decoding, each model performs a test generation on:

> Explain why transformers use self-attention.


In [ ]:
!pip install -q transformers accelerate sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

assert torch.cuda.device_count() >= 2, 'Need 2 GPUs'

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN')
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")

if HF_TOKEN is None:
    raise RuntimeError('Missing Kaggle secret HF_TOKEN')

target_name = 'meta-llama/Llama-3.2-3B'
draft_name  = 'meta-llama/Llama-3.2-1B'

tokenizer = AutoTokenizer.from_pretrained(target_name, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

target_model = AutoModelForCausalLM.from_pretrained(
    target_name,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
    device_map={'': 0}
)

draft_model = AutoModelForCausalLM.from_pretrained(
    draft_name,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
    device_map={'': 1}
)

target_model.eval()
draft_model.eval()

print('Loaded target on', next(target_model.parameters()).device)
print('Loaded draft on', next(draft_model.parameters()).device)

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Loaded target on cuda:0
Loaded draft on cuda:1


In [ ]:
@torch.inference_mode()
def generate(model, device, prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def strip_prompt(prompt, full):
    return full[len(prompt):] if full.startswith(prompt) else full

## Test run on each model

In [ ]:
prompt = 'Explain why transformers use self-attention.'

print('=== Target model ===')
t = generate(target_model, torch.device('cuda:0'), prompt, 200)
print(strip_prompt(prompt, t))

print('\n=== Draft model ===')
d = generate(draft_model, torch.device('cuda:1'), prompt, 200)
print(strip_prompt(prompt, d))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Target model ===
 What are the advantages of using self-attention over other attention mechanisms? How does self-attention work in practice? What are the limitations of self-attention?

=== Draft model ===
 The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential dat

## Speculative decoding (text‑only communication)

In [ ]:
def lcp(a, b):
    i = 0
    n = min(len(a), len(b))
    while i < n and a[i] == b[i]:
        i += 1
    return a[:i]

@torch.inference_mode()
def speculative_decode(prompt, max_new_tokens=256, draft_step=32):
    accepted = ''
    total = 0
    round_num = 0

    while total < max_new_tokens:
        round_num += 1
        ctx = prompt + accepted

        draft_full = generate(draft_model, torch.device('cuda:1'), ctx, draft_step)
        draft_text = strip_prompt(ctx, draft_full)

        print(f"\n--- Round {round_num} ---")
        print(f"Draft text generated: '{draft_text}'")

        if not draft_text:
            break

        draft_ids = tokenizer(draft_text, return_tensors='pt')['input_ids']
        need = draft_ids.shape[1]


        target_full = generate(target_model, torch.device('cuda:0'), ctx, need)
        target_text = strip_prompt(ctx, target_full)

        agreed = lcp(draft_text, target_text)

        if not agreed:
            fallback = generate(target_model, torch.device('cuda:0'), ctx, 8)
            fb = strip_prompt(ctx, fallback)
            accepted += fb
            total += tokenizer(fb, return_tensors='pt')['input_ids'].shape[1]
            print(f"No agreement. Falling back. Accepted {len(fb)} characters: '{fb}'")
        else:
            accepted += agreed
            total += tokenizer(agreed, return_tensors='pt')['input_ids'].shape[1]
            print(f"LCP agreed on {len(agreed)} characters: '{agreed}'")

        print(f"Total accepted so far: '{accepted}'")

    return prompt + accepted

In [ ]:
result = speculative_decode(prompt)
print(result)


--- Round 1 ---
Draft text generated: ' The transformer is a neural network architecture that is used to model sequential data. The transformer is a neural network architecture that is used to model sequential data. The transformer'
LCP agreed on 1 characters: ' '
Total accepted so far: ' '

--- Round 2 ---
Draft text generated: '1. The transformer is a new type of neural network architecture that has been proposed in recent years. The transformer is a new type of neural network architecture that has'
No agreement. Falling back. Accepted 46 characters: '2. Explain why transformers use positional enc'
Total accepted so far: ' 2. Explain why transformers use positional enc'

--- Round 3 ---
Draft text generated: 'odings. 3. Explain why transformers use self-attention. 4. Explain why transformers use positional encodings. 5. Explain why transformers use self'
LCP agreed on 40 characters: 'odings. 3. Explain why transformers use '
Total accepted so far: ' 2. Explain why transformers use p